In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from vivarium import Artifact

!date

Wed Oct  8 14:50:57 PDT 2025


In [2]:
!ls -halt washington.hdf  # expects this to be in root directory of repo

-rw-r--r-- 1 abie abie 62M Apr  3  2025 washington.hdf


In [3]:
%run src/vivarium_nih_moud/data/dismod_at.py

AssertionError: ti_male_ should have some data

In [ ]:
art = Artifact(path='washington.hdf')

In [ ]:
def art_etl(key):
    data = art.load(key)
    data = (data.apply(pd.DataFrame.describe, percentiles=[0.025, 0.975], axis=1)
            .filter(['mean', '2.5%', '97.5%']).reset_index())
    data = data[data.age_end <= 95]
    return data

In [ ]:
def age_group_plot(measure, art_old, art_new):
    fig, ax = plt.subplots(ncols=2, sharex=True, sharey=True, figsize=(12, 5))
    for j, sex in enumerate(art_new.sex.unique()):
        for i, art_data in enumerate([art_old, art_new]):
            color = f'C{i}'
            if i == 0:
                linestyle = '--'
                label='Old'
            else:
                linestyle = '-'
                label='New'

            art_plot = art_data.loc[(art_data.sex == sex)]

            ax[j].fill_between(art_plot.age_start, art_plot['2.5%'], art_plot['97.5%'], alpha=0.2, color=color)
            ax[j].plot(art_plot.age_start, art_plot['mean'], linestyle=linestyle, label=label, color=color, linewidth=3)


        ax[j].text(.5, .95, f"{sex}", va='top', ha='center', transform=ax[j].transAxes)
        ax[j].set_xlabel("Age (years)")

        if j == 0:
            ax[j].set_ylabel(f"{measure}")
        if j == 1:
            ax[j].legend(loc=(1.01, 0))

        plt.subplots_adjust(wspace=0)

In [ ]:
# Plot ODE errors
df_new = art_etl('cause.oud_consistent.ode_errors')
df_old = df_new.iloc[[], :]
age_group_plot('ode_errors', df_old, df_new)

In [ ]:
# Plot prevalence
df_old = art_etl('cause.opioid_use_disorders.prevalence')
df_new = art_etl('cause.oud_consistent.prevalence')
age_group_plot('prevalence', df_old, df_new)

In [ ]:
# Plot incidence rate
df_old = art_etl('cause.opioid_use_disorders.incidence_rate')
df_new = art_etl('cause.oud_consistent.incidence_rate')
age_group_plot('incidence_rate', df_old, df_new)

In [ ]:
# Plot excess mortality rate
df_old = art_etl('cause.opioid_use_disorders.excess_mortality_rate')
df_new = art_etl('cause.oud_consistent.excess_mortality_rate')
age_group_plot('excess_mortality_rate', df_old, df_new)

In [ ]:
# Plot treatment ratio
df_old = art_etl('cause.oud_consistent.treatment_ratio')
df_new = art_etl('cause.oud_consistent.treatment_ratio')
age_group_plot('treatment_ratio', df_old, df_new)

In [ ]:
# Plot remission rate
df_old = art_etl('cause.oud_consistent.remission_rate')
df_new = art_etl('cause.oud_consistent.remission_rate')
age_group_plot('remission_rate', df_old, df_new)

In [ ]:
# Plot treatment initiation rate
df_old = art_etl('cause.oud_consistent.treatment_initiation_rate')
df_new = art_etl('cause.oud_consistent.treatment_initiation_rate')
age_group_plot('treatment_initiation_rate', df_old, df_new)

In [ ]:
# Plot treatment success rate
df_old = art_etl('cause.oud_consistent.treatment_success_rate')
df_new = art_etl('cause.oud_consistent.treatment_success_rate')
age_group_plot('treatment_success_rate', df_old, df_new)

In [ ]:
# Plot treatment failure rate
df_old = art_etl('cause.oud_consistent.treatment_failure_rate')
df_new = art_etl('cause.oud_consistent.treatment_failure_rate')
age_group_plot('treatment_failure_rate', df_old, df_new)

In [ ]:
# Visualize sample variation in treatment ratio across age
t = art.load('cause.oud_consistent.treatment_ratio')
plt.figure(figsize=(12, 5))
for col in t.filter(like='draw_').columns[:25]:
    plt.plot(t[col].unstack(level=0).reset_index().set_index('age_start').Male, color='C1', alpha=.5)
    plt.plot(t[col].unstack(level=0).reset_index().set_index('age_start').Female, color='C0', alpha=.5)
plt.xlabel('Age (years)')
plt.ylabel('Treatment Ratio')
plt.title('Treatment Ratio Sample Draws (Male=orange, Female=blue)')